# R-Learner Demo

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path

import numpy as np
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.kernel_ridge import KernelRidge
from xgboost import XGBClassifier, XGBRegressor

from rlearner import (
    CrossFittedNuisanceEstimator,
    RLearner,
    RLossStacking,
    SuperLearnerClassifier,
    SuperLearnerRegressor,
)

SEED = 42


In [3]:
data_path = Path("data.csv")
if not data_path.exists():
    raise FileNotFoundError("data.csv was not found in the current directory.")

df = pd.read_csv("data.csv")

Y = df["y"].values
W = df["t"].values
ITE = df["ite"].values

X = df.drop(columns=["y", "t", "ite"]).values

print(f"n = {len(df):,}, p = {X.shape[1]}")


n = 10,000, p = 20


## Step 1: Nuisance Estimation with Constrained Super Learners

In [ ]:
outcome_model = SuperLearnerRegressor(
    estimators=[
        ("rf", RandomForestRegressor(random_state=SEED, n_jobs=-1)),
        ("xgb", XGBRegressor(
            objective="reg:squarederror",
            random_state=SEED,
            n_jobs=-1,
            tree_method="hist",
            verbosity=0,
        )),
    ],
    estimator_param_grids={
        "rf": {
            "n_estimators": [300],
            "max_depth": [8, None],
            "min_samples_leaf": [5, 20],
        },
        "xgb": {
            "n_estimators": [300],
            "max_depth": [3, 5],
            "learning_rate": [0.05, 0.1],
            "subsample": [0.8],
            "colsample_bytree": [0.8],
        },
    },
    search_cv=10,
    search_n_jobs=-1,
    random_state=SEED,
    normalize_weights=True,
)

treatment_model = SuperLearnerClassifier(
    estimators=[
        ("rf", RandomForestClassifier(random_state=SEED, n_jobs=-1)),
        ("xgb", XGBClassifier(
            objective="binary:logistic",
            random_state=SEED,
            n_jobs=-1,
            tree_method="hist",
            eval_metric="logloss",
            verbosity=0,
        )),
    ],
    estimator_param_grids={
        "rf": {
            "n_estimators": [300, 500],
            "max_depth": [6, None],
            "min_samples_leaf": [5, 20],
        },
        "xgb": {
            "n_estimators": [200, 400],
            "max_depth": [3, 5],
            "learning_rate": [0.05, 0.1],
            "subsample": [0.8],
            "colsample_bytree": [0.8],
        },
    },
    search_cv=3,
    search_n_jobs=-1,
    random_state=SEED,
    normalize_weights=True,
)

nuisance = CrossFittedNuisanceEstimator(
    outcome_model=outcome_model,
    treatment_model=treatment_model,
    n_folds=10,
    random_state=SEED,
    refit_full=True,
)

nuisance_result = nuisance.fit_predict(X, Y, W)
print(nuisance_result.diagnostics)


{'outcome_rmse': 1.1988369293033962, 'propensity_auc': 0.6790551273061427, 'propensity_log_loss': 0.6442788659215452}


In [4]:
outcome_weights = nuisance.get_outcome_weights()
treatment_weights = nuisance.get_treatment_weights()

print("Outcome stacking weights:")
display(pd.DataFrame({
    "learner": list(outcome_weights.keys()),
    "weight": list(outcome_weights.values()),
}))

print("Treatment stacking weights:")
display(pd.DataFrame({
    "learner": list(treatment_weights.keys()),
    "weight": list(treatment_weights.values()),
}))


Outcome stacking weights:


,learner,weight
0,rf,1.000000e+00
1,xgb,2.109402e-17


Treatment stacking weights:


,learner,weight
0,rf,1.0
1,xgb,0.0


## Step 2: Second-Stage R-Loss Learners

In [9]:
second_stage_learners = {
    "rf": RandomForestRegressor(
        n_estimators=500,
        max_features=0.75,
        min_samples_leaf=10,
        max_samples=0.8,
        n_jobs=-1,
        random_state=SEED,
        bootstrap=True,
    ),
    "kr": KernelRidge(alpha=1.0, kernel="rbf", gamma=0.1),
}

rlearner = RLearner(
    cate_learners=second_stage_learners,
    stacking_model=RLossStacking(lambda_reg=1.0),
)

tau_hat = rlearner.fit_predict(
    X,
    Y,
    W,
    y_hat=nuisance_result.y_hat,
    d_hat=nuisance_result.d_hat,
)

print(f"a_hat = {rlearner.stacking_model_.a_hat:.6f}")
print(f"b_hat = {rlearner.stacking_model_.b_hat:.6f}")

r_loss_weights = pd.DataFrame({
    "learner": rlearner.learner_names_,
    "alpha_hat": rlearner.stacking_model_.alpha_hat,
})

print("Second-stage R-loss stacking weights (alpha_hat):")
display(r_loss_weights)

rmse_tau = np.sqrt(np.mean((ITE - tau_hat) ** 2))
corr_tau = np.corrcoef(ITE, tau_hat)[0, 1]
print(f"RMSE of CATE predictions: {rmse_tau:.4f}")
print(f"Correlation between true ITE and predicted CATE: {corr_tau:.4f}")


a_hat = 1.611920
b_hat = 0.899480
Second-stage R-loss stacking weights (alpha_hat) and coefficients (beta_hat):


,learner,alpha_hat,beta_hat
0,rf,0.957387,0.861150
1,kr,0.288808,0.259777


RMSE of CATE predictions: 0.4832
Correlation between true ITE and predicted CATE: 0.6066


## Step 3: Evaluation Tests


In [6]:
blp = rlearner.blp_test(confidence_level=0.95)
blp_summary = pd.DataFrame({
    "parameter": ["alpha", "beta"],
    "estimate": [blp.alpha.estimate, blp.beta.estimate],
    "hc2_se": [blp.alpha.std_error, blp.beta.std_error],
    "z_value": [blp.alpha.z_value, blp.beta.z_value],
    "p_value": [blp.alpha.p_value, blp.beta.p_value],
    "ci_lower": [blp.alpha.confidence_interval[0], blp.beta.confidence_interval[0]],
    "ci_upper": [blp.alpha.confidence_interval[1], blp.beta.confidence_interval[1]],
})

print("BLP test summary:")
display(blp_summary)


BLP test summary:


,parameter,estimate,hc2_se,z_value,p_value,ci_lower,ci_upper
0,alpha,-0.003678,0.075394,-0.048786,0.96109,-0.151448,0.144092
1,beta,1.001846,0.037337,26.832321,0.00000,0.928667,1.075026


In [7]:
calibration = rlearner.calibration_test(n_bins=5)
print(f"CAL_1 = {calibration.cal_l1:.6f}")
print(f"CAL_2 = {calibration.cal_l2:.6f}")

calibration_bins = pd.DataFrame([
    {
        "bin": row.bin_index,
        "size": row.size,
        "theta_star": row.theta_star,
        "theta_dr": row.theta_dr,
        "gap": row.gap,
        "tau_min": row.tau_min,
        "tau_max": row.tau_max,
    }
    for row in calibration.bins
])
display(calibration_bins)


CAL_1 = 665.084625
CAL_2 = 48.885481


,bin,size,theta_star,theta_dr,gap,tau_min,tau_max
0,1,2000,1.343496,1.419654,0.076158,0.347318,1.616348
1,2,2000,1.755106,1.721197,-0.033909,1.616368,1.881092
2,3,2000,1.989993,1.938714,-0.051279,1.881105,2.095829
3,4,2000,2.212464,2.137095,-0.075370,2.095842,2.339343
4,5,2000,2.568048,2.663875,0.095826,2.339380,3.556674


In [8]:
uplift = rlearner.uplift_test()
print(f"AUUC = {uplift.auuc:.6f}")

uplift_curve = pd.DataFrame({
    "fraction": uplift.fractions,
    "size": uplift.subgroup_sizes,
    "theta_dr": uplift.theta_dr,
})
display(uplift_curve)


AUUC = 2.101576


,fraction,size,theta_dr
0,0.1,1000,2.917842
1,0.2,2000,2.663875
2,0.3,3001,2.519490
3,0.4,4000,2.410358
4,0.5,5000,2.330558
5,0.6,6000,2.257943
6,0.7,7001,2.193099
7,0.8,8000,2.126354
8,0.9,9000,2.058894
9,1.0,10000,1.992539
